# CS336 Spring 2026 - Pre-tokenization
----
> 目的是在正式 Tokenizer 之前加入 Pre-tokenization

整体路线变为:
```text
raw text
    ↓
pre-tokenization
    ↓
很多局部 pieces
    ↓
每个 piece 内部独立做 BPE
```
> Pre-tokenization 的目的并不是 "先把文本分成单词", 而是定义 BPE 的 merge boundary.

## 1. 为什么需要 Pre-tokenization?
因为上一节输入的 `the cat in the hat`, Toy BPE 会看到:
```text
t h e ␠ c a t ␠ i n ␠ t h e ␠ h a t
```
也就是说 `e ␠` 也是合法的 pair, 所以 `(e, space)` 可以参与统计和合并. 但是加入 pre-tokenization 之后, 可能会变为:
```text
["the", " cat", " in", " the", " hat"]
```
其内部结构为:
```text
[t h e]

[␠ c a t]

[␠ i n]

[␠ t h e]

[␠ h a t]
```
方括号其实就是 BPE boundary. 因此 `e | ␠` 位于两个不同的 pre-token, 所以不可以 merge. 但是 `(space, c)` 仍然允许, 因为 `" cat"` 本身就是一个 pre-token.
> GPT-style tokenizer 很容易学习 `" cat"`, `" world"`, `" the"` 这种带前导空格的 token.

## 2. 预分词在做什么?
预分词是在 BPE 之前, 先把文本切成一些初步的片段, 比如:
- 单词: `hello`
- 数字: `123`
- 标点: `!`, `\`
- 空格和换行: ` `
- 特殊token: `<|endoftext|>`
这样做的目的是**避免 BPE 跨边界合并**.

> 在实现过程中选用 `regex` 而不是 Python 的`re`. 因为后续会使用 `\p{L} \p{N}` 这种 Unicode property, 而 Python 的 `re` 模块不支持这些 Unicode property.

In [1]:
import regex

print(regex.__name__)

regex


## 3. GPT-2-style Pre-Tokenization regex
先使用一个与 GPT-2 的原始 split pattern 对应的 pattern. OpenAI 当前在 `tiktoken` 中还维护了一个语义等价, 执行更快的 `r50` 版本.

In [2]:
GPT2_PRETOKEN_PATTERN = (
    r"""'(?:[sdmt]|ll|ve|re)"""  # 英语缩写
    r"""| ?\p{L}+"""  # 可选空格+字母
    r"""| ?\p{N}+"""  # 可选空格+数字
    r"""| ?[^\s\p{L}\p{N}]+"""  # 可选空格+标点/符号
    r"""|\s+(?!\S)"""  # 末尾空白
    r"""|\s+"""  # 其他空白
)


def pretokenize(text: str) -> list[str]:
    # 预分词

    return [match.group(0) for match in regex.finditer(GPT2_PRETOKEN_PATTERN, text)]


上述 `GPT2_PRETOKEN_PATTERN` 的解释:
1. 匹配英语中的缩写后缀, 如 `'s, 't, 'd, 'm, 'll, 've', 're'`. 例如 `It's` 会被切为 `It'` 和 `'s` 两个片段.
2. 匹配可选的一个空格, 后面跟一个或多个 Unicode 字母. 目的是将前导空格附着到单词上, 例如 ` Hello`.
3. 类似上述, 后面可以跟一个或多个 Unicode 数字.
4. 匹配可选空格 + 一个或多个非空白, 非字母, 非数字的字符, 即标点, 符号, emoji等.
5. 匹配一个或多个空白字符, 但要求后面不再有非空白字符. 这用于捕获字符串末尾的空白, 避免其被后面的分支拆开.
6. 当上面条件不满足时, 匹配一个或多个空白字符(用于兜底).

## 4. 第一个实验: `Hello, World!`

In [3]:
text = "Hello, world!"

pieces = pretokenize(text)

print("=====预分词=====")
print(pieces)
print("reconstructed:", "".join(pieces))
print("round trip:", "".join(pieces) == text)

=====预分词=====
['Hello', ',', ' world', '!']
reconstructed: Hello, world!
round trip: True


注意上述的 ` world` 是一个完整的 pre-token. 所以

<center>
space + w + o + r + l + d
</center>

后续的 BPE merge 被允许在此范围中参与.

## 5. 第一个重要的不变量
在 Pre-tokenization 之后, 希望
```python
"".join(pieces) == text
```
也始终成立, 也就是满足:

<center>
join(pretokenize(text)) = text
</center>

因为 tokenizer 不可以
- 删除空格
- 删除换行
- 修改标点
- 修改 Unicode
否则最终会无法严格恢复原始文本.

## 6. 研究 `Hello Hello`

In [4]:
text = "Hello Hello"
pieces = pretokenize(text)

for i, piece in enumerate(pieces):
    print(i, repr(piece))

0 'Hello'
1 ' Hello'


上述这两个 Hello 其实已经是两个不同的 byte sequences.
1. 第一个 `"Hello".encode("utf-8")` 的结果是 `H e l l o`.
2. 第二个 `" Hello".encode("utf-8")` 的结果是 `space H e l l o`

## 7. 观察 Contractions
`regex` 的第一部分:
```text
'(?:[sdmt]|ll|ve|re)
```

In [5]:
text = "I'm don't we'll they've"

print(pretokenize(text))

['I', "'m", ' don', "'t", ' we', "'ll", ' they', "'ve"]


这可以看到:
> pre-tokenization 不是"按照英文单词切词", 而是在设置 BPE merge boundaries.

## 8. 观察 Letters, Numbers 和 Punctuation

In [6]:
text = "I'm learning CS336."
pieces = pretokenize(text)

for piece in pieces:
    print(repr(piece))


'I'
"'m"
' learning'
' CS'
'336'
'.'


上述结果分别来自
```text
'I' → letters

"'m" → contraction

' learning' → optional space + letters

' CS' → optional space + letters

'336' → numbers

'.' → punctuation
```

## 9. 为什么不能写 `[A-Za-z]+`
下面尝试 Unicode

In [7]:
samples = [
    "hello",
    "你好世界",
    "こんにちは世界",
    "café",
    "Привет",
]

for text in samples:
    print(repr(text), "->", pretokenize(text))


'hello' -> ['hello']
'你好世界' -> ['你好世界']
'こんにちは世界' -> ['こんにちは世界']
'café' -> ['café']
'Привет' -> ['Привет']


因为 `\p{L}` 不是 English letters, 而是 Unicode Letter category. 所以这是一个 Unicode-aware tokenizer.

## 10. 考虑 Emoji
考虑 `🌍`, 它既不是 `\p{L}`, 也不是 `\p{N}`, 更不是 whitespace. 因此会进入下面的
```text
[^\s\p{L}\p{N}]+
```

In [8]:
text = "Hello 你好 🌍!"
pieces = pretokenize(text)

for piece in pieces:
    print(repr(piece), "->", list(piece.encode("utf-8")))


'Hello' -> [72, 101, 108, 108, 111]
' 你好' -> [32, 228, 189, 160, 229, 165, 189]
' 🌍!' -> [32, 240, 159, 140, 141, 33]


上述值得注意的结果是: ` 🌍!` 可能整体成为一个 pre-token.
> 所以 pre-token 不一定是 word. 它只是允许 BPE 在里面进行 merge 的局部区域.

## 11. 为什么不直接用 `text.split()`?
比如下面的例子, 在 `Hello,  world!` 中, 有两个空格, 还有 `\n` 字符, 而 `text.split()` 会将这些字符信息丢掉, 从而无法正确的恢复原始文本.

In [9]:
text = "Hello,  world!\nI'm learning CS336."

print("str.split():")
print(text.split())

print("\nregex pre-tokenization:")
print(pretokenize(text))

print("\nsplit() 是否能正确重建原始字符串?")
print(" ".join(text.split()) == text)

print("regex pieces 是否能正确重建原始字符串?")
print("".join(pretokenize(text)) == text)

str.split():
['Hello,', 'world!', "I'm", 'learning', 'CS336.']

regex pre-tokenization:
['Hello', ',', ' ', ' world', '!', '\n', 'I', "'m", ' learning', ' CS', '336', '.']

split() 是否能正确重建原始字符串?
False
regex pieces 是否能正确重建原始字符串?
True


## 12. 回到 `the cat in the hat`


In [10]:
text = "the cat in the hat"
pieces = pretokenize(text)

print(pieces)

['the', ' cat', ' in', ' the', ' hat']


对于此输出, Toy BPE 看到:
```text
t h e ␠ c a t ␠ i n ␠ t h e ␠ h a t
```
而 Pre-Tokenization 看到:
```text
[t h e]

[␠ c a t]

[␠ i n]

[␠ t h e]

[␠ h a t]
```
于是 BPE 只能在 `[ ]` 内部进行 merge.

## 13. `Regex` 和 `BPE` 的比较
Pre-Tokenization regex 工作在 `Python str` 和 `Unicode` 上. 其完整顺序是:
```text
Unicode string
      ↓
regex pre-tokenization
      ↓
Unicode pre-token strings
      ↓
UTF-8
      ↓
byte sequences
      ↓
BPE
```

## 14. 将 pre-token 转为 bytes.

In [11]:
def pretoken_byte_sequences(text: str) -> list[tuple[int, ...]]:
    # 将 pre-token 转化为字节序列

    return [tuple(piece.encode("utf-8")) for piece in pretokenize(text)]

In [12]:
byte_sequences = pretoken_byte_sequences("the cat in the hat")

for piece, sequence in zip(pretokenize("the cat in the hat"), byte_sequences):
    print(repr(piece), "->", sequence)


'the' -> (116, 104, 101)
' cat' -> (32, 99, 97, 116)
' in' -> (32, 105, 110)
' the' -> (32, 116, 104, 101)
' hat' -> (32, 104, 97, 116)


## 15. Pre-Tokenization 阻止了什么?
现在做对比实验

In [13]:
from collections import Counter


# 定义pair counting 函数
def count_pairs_in_one_sequence(sequence: tuple[int, ...]) -> Counter[tuple[int, int]]:

    return Counter(zip(sequence, sequence[1:]))

In [14]:
# Toy BPE
text = "the cat in the hat"

giant_sequence = tuple(text.encode("utf-8"))

toy_counts = count_pairs_in_one_sequence(giant_sequence)

In [15]:
# Pre-Tokenization
bounded_counts: Counter[tuple[int, int]] = Counter()

for sequence in pretoken_byte_sequences(text):
    bounded_counts.update(count_pairs_in_one_sequence(sequence))


In [16]:
e_space = (ord("e"), ord(" "))
space_c = (ord(" "), ord("c"))

print("(e, space)")
print("Toy BPE:", toy_counts[e_space])
print("Pre-tokenized BPE:", bounded_counts[e_space])

print("\n(space, c)")
print("Toy BPE:", toy_counts[space_c])
print("Pre-tokenized BPE:", bounded_counts[space_c])

(e, space)
Toy BPE: 2
Pre-tokenized BPE: 0

(space, c)
Toy BPE: 1
Pre-tokenized BPE: 1


上述的 `(e, space)` 消失了.

因为 `the cat` 被分成 `"the" " cat"` 了. 

所以 `(e, space)` 跨过了边界, 从而不合法.而 `(space, c)` 位于 `" cat"` 内部, 因此是合法的.

## 16. Toy BPE 的第三轮发生了变化
Toy BPE 前三轮为:
```text
t + h → th

th + e → the

the + space → "the "
```
而加入 pre-tokenization 后, `(the, space)` 是跨 pre-token 边界的, 所以不是合法的 merge.
> 因此 Pre-Tokenization 会改变 BPE 最终学到的 vocabulary. 它直接限定 BPE search space.

## 17. 效率问题
下面假设 corpus 很大, 里面的 `" the"` 出现了 500000 次. 但是完全没有必要保存 500000 个. 因为对于 BPE training, 真正关系的是 pre-token 是什么以及它出现的频率. 所以可以保存:

<center>
pre-token -> frequency
</center>

## 18. `pre-token -> frequency`

In [17]:
def count_pretokens(text: str) -> Counter[tuple[int, ...]]:
    counts: Counter[tuple[int, ...]] = Counter()

    for piece in pretokenize(text):
        byte_sequence = tuple(piece.encode("utf-8"))

        counts[byte_sequence] += 1

    return counts

In [18]:
corpus = "the cat and the cat\nThe cat."
counts = count_pretokens(corpus)

for byte_seq, frequency in counts.most_common():
    # print(bytes(byte_seq).decode("utf-8"), "->", frequency) 这样会将 `\n` 当作换行符进行转义, 导致输出不正确
    print(repr(bytes(byte_seq).decode("utf-8")), "->", frequency)


' cat' -> 3
'the' -> 1
' and' -> 1
' the' -> 1
'\n' -> 1
'The' -> 1
'.' -> 1


### 这里使用 `tuple[int, ...]` 而不是 `list`
主要问题是 `list` 不能作为 `dict Counter` 的键, 而 `tuple` 可以.

In [19]:
example = tuple(b" cat")

demo = Counter()

demo[example] += 1
demo[example] += 1

print(example)
print(demo)

(32, 99, 97, 116)
Counter({(32, 99, 97, 116): 2})


## 19. `pretoken frequency` 如何影响 pair frequency?
假设:
```text
pre-token: " cat"

frequency: 3
```
对于 byte sequence: `␠ c a t`

内部的 pairs:
```text
(␠, c)
(c, a)
(a, t)
```
由于这个 pre-token 的频率是 3, 所以这些 pairs 对全局 frequency 的贡献分别都是 3, 而不是1.

In [20]:
def count_weighted_pairs(
    pretoken_counts: Counter[tuple[int, ...]],
) -> Counter[tuple[int, int]]:
    pair_counts: Counter[tuple[int, int]] = Counter()

    for sequence, frequency in pretoken_counts.items():
        for pair in zip(sequence, sequence[1:]):
            pair_counts[pair] += frequency

    return pair_counts

In [21]:
pair_counts = count_weighted_pairs(counts)

interesting_pairs = [(ord(" "), ord("c")), (ord("c"), ord("a")), (ord("a"), ord("t"))]

for pair in interesting_pairs:
    readable_pair = (bytes([pair[0]]).decode("utf-8"), bytes([pair[1]]).decode("utf-8"))

    print(readable_pair, "->", pair_counts[pair])


(' ', 'c') -> 3
('c', 'a') -> 3
('a', 't') -> 3


## 20. 正式的 BPE 数据结构
上一节的 trainer 是 one token sequence 的, 现在已经是 `Counter[tuple[int, ...]]`, 概念上有:
```text
{
    (32, 99, 97, 116): 3,      # " cat"

    (116, 104, 101): 1,        # "the"

    (32, 116, 104, 101): 1,    # " the"

    ...
}
```
这比 "把 corpus 中所有的 pre-token 全部保存一遍" 合理的多.

## 21. 现在的数据流
目前的流程是:
```text
corpus
    ↓
regex.finditer(...)
    ↓
pre-token strings
    ↓
UTF-8 encode
    ↓
tuple[int, ...]
    ↓
Counter
    ↓
pretoken → frequency
```
接下来的 pair counting:
```text
pretoken → frequency
        ↓
遍历每个 pre-token 内部 adjacent pairs
        ↓
pair count × pretoken frequency
        ↓
global pair_counts
```
然后才:
```text
选 frequency 最大 pair
↓
BPE merge
```

## 22. 区分三种 frequency
### (1) Corpus frequency
`" cat"` 整个 corpus 出现 3 次.

### (2) Local pair occurrence
在一次 `" cat"` 内部, `(c, a)` 出现 1 次.

### (3) Global pair frequency
因为 `" cat"` 出现 3 次, 所以:
$$
count(c, a) = 1 \times 3 = 3
$$
一般来说:
$$
global\ pair\ count = \sum_w local\ count(pair, w)\times freq(w)
$$
其中: `w = pre-token`.

## 23. 最后检查

In [22]:
test_strings = [
    "",
    "Hello, world!",
    "hello hello",
    "I'm learning CS336.",
    "你好世界",
    "Hello 你好 🌍!",
    "a  b\nc",
]

for text in test_strings:
    assert "".join(pretokenize(text)) == text


In [23]:
pieces = pretokenize("the cat in the hat")

assert pieces == ["the", " cat", " in", " the", " hat"]

print("pretokenize passed")

pretokenize passed


In [24]:
toy_counts = count_pairs_in_one_sequence(tuple(b"the cat in the hat"))

bounded_counts = Counter()

for sequence in pretoken_byte_sequences("the cat in the hat"):
    bounded_counts.update(count_pairs_in_one_sequence(sequence))


In [25]:
assert toy_counts[(ord("e"), ord(" "))] == 2
assert bounded_counts[(ord("e"), ord(" "))] == 0

print("boundary passed")

boundary passed


In [26]:
corpus_counts = count_pretokens("the cat and the cat\nThe cat.")

assert corpus_counts[tuple(b" cat")] == 3

print("frequency passed")

frequency passed


In [27]:
weighted = count_weighted_pairs(corpus_counts)

assert weighted[(ord(" "), ord("c"))] == 3
assert weighted[(ord("c"), ord("a"))] == 3

print("weighted passed")

weighted passed


## 24. 目前的结构
```text
text
    ↓
regex pre-tokenization
    ↓
pre-token strings
    ↓
UTF-8
    ↓
byte tuples
    ↓
pretoken → frequency
    ↓
pair counting inside each pre-token
    ↓
weighted global pair frequency
    ↓
merge inside pre-token boundaries only
```